# Unified Data Cube Creation

This notebook creates a **denormalized, unified dataset** by joining the fact table with all dimension tables.

## Purpose:
Create a single dataset that can answer all 10 KPIs without repeated joins

## Output:
1. **Delta Table**: `subscription_pipeline.gold.data_cube`
2. **CSV File**: Exported to data_cube folder
3. **Excel File**: Exported to data_cube folder

## Data Cube Structure:
- Fact metrics (revenue, dates, flags)
- Customer attributes (name, country, industry, type)
- Product attributes (name, plan, billing cycle)
- Date attributes (quarter, financial year)
- Country details
- Employee/sales owner details

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

print("Loading all tables...")
print("="*70)

# Load fact table
fact = spark.read.table("subscription_pipeline.gold.fact_subscription")
print(f"✓ Fact table loaded: {fact.count():,} rows")

# Load dimension tables
dim_customer = spark.read.table("subscription_pipeline.gold.dim_customer")
print(f"✓ dim_customer loaded: {dim_customer.count():,} rows")

dim_product = spark.read.table("subscription_pipeline.gold.dim_product")
print(f"✓ dim_product loaded: {dim_product.count():,} rows")

dim_country = spark.read.table("subscription_pipeline.gold.dim_country")
print(f"✓ dim_country loaded: {dim_country.count():,} rows")

dim_employee = spark.read.table("subscription_pipeline.gold.dim_employee")
print(f"✓ dim_employee loaded: {dim_employee.count():,} rows")

dim_date = spark.read.table("subscription_pipeline.gold.dim_date")
print(f"✓ dim_date loaded: {dim_date.count():,} rows")

print("\n✓ All tables loaded successfully!")

In [0]:
print("\nBuilding unified data cube...")
print("="*70)

# Start with fact table and join all dimensions
data_cube = fact \
    .join(
        dim_customer.select(
            col("customer_id"),
            col("customer_name"),
            col("country").alias("customer_country"),
            col("industry_type"),
            col("account_created_date"),
            col("is_active").alias("customer_is_active"),
            col("customer_type")
        ),
        "customer_id",
        "left"
    ) \
    .join(
        dim_product.select(
            col("product_id"),
            col("product_name"),
            col("plan_name"),
            col("billing_cycle")
        ),
        "product_id",
        "left"
    ) \
    .join(
        dim_country.select(
            col("country_code"),
            col("country_name"),
            col("currency_code")
        ),
        col("customer_country") == col("country_code"),
        "left"
    ) \
    .join(
        dim_date.select(
            col("date"),
            col("quarter"),
            col("financial_year"),
            col("week").alias("week_of_year")
        ),
        fact["start_date"] == dim_date["date"],
        "left"
    ) \
    .drop("date", "country_code")

print("✓ All dimensions joined to fact table")
print(f"✓ Data cube created: {data_cube.count():,} rows")
print(f"✓ Total columns: {len(data_cube.columns)}")

# Display sample
print("\nSample Data (first 5 rows):")
display(data_cube.limit(5))

In [0]:
print("\nAdding computed metrics...")
print("="*70)

# Add useful computed columns
data_cube = data_cube \
    .withColumn(
        "subscription_duration_days",
        when(col("end_date").isNotNull(), 
             datediff(col("end_date"), col("start_date"))
        ).otherwise(None)
    ) \
    .withColumn(
        "is_active_subscription",
        when(col("end_date").isNull(), True).otherwise(False)
    ) \
    .withColumn(
        "year_month",
        concat(col("year"), lit("-"), lpad(col("month"), 2, "0"))
    ) \
    .withColumn(
        "revenue_category",
        when(col("revenue_gbp") >= 10000, "High")
        .when(col("revenue_gbp") >= 5000, "Medium")
        .when(col("revenue_gbp") >= 1000, "Low")
        .otherwise("Very Low")
    ) \
    .withColumn(
        "customer_tenure_years",
        round(datediff(col("start_date"), col("account_created_date")) / 365.25, 1)
    )

print("✓ Computed metrics added")
print(f"✓ Final data cube: {data_cube.count():,} rows, {len(data_cube.columns)} columns")

In [0]:
print("\nData Cube Schema:")
print("="*70)
data_cube.printSchema()

print("\nColumn Summary:")
print("="*70)
for i, col_name in enumerate(data_cube.columns, 1):
    print(f"{i:2d}. {col_name}")

In [0]:
print("\nSaving data cube as Delta table...")
print("="*70)

# Save to catalog
data_cube.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("subscription_pipeline.gold.data_cube")

print("✓ Data cube saved as: subscription_pipeline.gold.data_cube")
print("\nYou can now query it directly in KPIs:")
print("  spark.read.table('subscription_pipeline.gold.data_cube')")

In [0]:
print("\nExporting data cube to CSV...")
print("="*70)

# Convert to Pandas for CSV export (limit to avoid memory issues)
data_cube_pandas = data_cube.toPandas()

# Save to CSV in workspace
csv_path = "/Workspace/Users/gauravrathore856@gmail.com/DE-mini-project-subscriptions/03_gold/data_cube/data_cube.csv"
data_cube_pandas.to_csv(csv_path, index=False)

print(f"✓ CSV exported to: {csv_path}")
print(f"✓ Size: {len(data_cube_pandas):,} rows × {len(data_cube_pandas.columns)} columns")

In [0]:
import pandas as pd

print("\nExporting data cube to Excel...")
print("="*70)

# Install openpyxl if needed
try:
    import openpyxl
except ImportError:
    print("Installing openpyxl...")
    %pip install openpyxl -q
    import openpyxl

# Export to Excel
excel_path = "/Workspace/Users/gauravrathore856@gmail.com/DE-mini-project-subscriptions/03_gold/data_cube/data_cube.xlsx"

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Main data sheet
    data_cube_pandas.to_excel(writer, sheet_name='Data_Cube', index=False)
    
    # Add summary sheet
    summary_data = {
        'Metric': [
            'Total Rows',
            'Total Columns',
            'Total Customers',
            'Total Products',
            'Date Range (Start)',
            'Date Range (End)',
            'Total Revenue',
            'Active Subscriptions',
            'Ended Subscriptions'
        ],
        'Value': [
            f"{len(data_cube_pandas):,}",
            len(data_cube_pandas.columns),
            data_cube_pandas['customer_id'].nunique(),
            data_cube_pandas['product_id'].nunique(),
            str(data_cube_pandas['start_date'].min()),
            str(data_cube_pandas['start_date'].max()),
            f"£{data_cube_pandas['revenue_gbp'].sum():,.2f}",
            data_cube_pandas['is_active_subscription'].sum(),
            (~data_cube_pandas['is_active_subscription']).sum()
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"✓ Excel exported to: {excel_path}")
print(f"✓ Sheets: Data_Cube (main data), Summary (statistics)")

## ✅ Data Cube Creation Complete!

### Output Files:
1. **Delta Table**: `subscription_pipeline.gold.data_cube`  
   → Use in all KPI calculations

2. **CSV**: `/Workspace/Users/.../03_gold/data_cube/data_cube.csv`  
   → For external analysis

3. **Excel**: `/Workspace/Users/.../03_gold/data_cube/data_cube.xlsx`  
   → For evaluation/presentation

### Next Steps:
- Update KPI notebooks to use `subscription_pipeline.gold.data_cube`
- Remove redundant joins in KPI calculations
- Run all KPIs to verify results

In [0]:
# Quick verification - test querying the data cube
print("\nVerification: Querying the data cube table...")
print("="*70)

cube_test = spark.read.table("subscription_pipeline.gold.data_cube")

print(f"✓ Table accessible: subscription_pipeline.gold.data_cube")
print(f"✓ Row count: {cube_test.count():,}")
print(f"✓ Column count: {len(cube_test.columns)}")

# Show sample aggregation
print("\nSample Query: Revenue by Year")
revenue_by_year = cube_test.groupBy("year").agg(
    sum("revenue_gbp").alias("total_revenue"),
    countDistinct("customer_id").alias("customers")
).orderBy("year")

display(revenue_by_year)

print("\n✓ Data cube is ready for KPI calculations!")

In [0]:
display(cube_test)